Selección y Armonización de Datos

1. Seleccionar datasets textuales etiquetados con licencia clara y tamaño >1GB.
2. Unificar el esquema de columnas de los datos (text, label).
3. Normalizar la taxonomía de etiquetas utilizando mapas de etiquetas.
4. Deduplicar textos para evitar la fuga de información entre train y test.
5. Controlar el desbalance de clases usando pesos por clase o sampling si es necesario.

Tarea Técnica: Modelado y Evaluación

6. Diseñar un pipeline de PLN de extremo a extremo para clasificación multiclase.
7. Implementar baselines clásicos como TF-IDF (+ BoW) con regresión logística o SVM lineal.
8. Realizar validación con CV o partición train/val/test, siempre con estratificación.
9. Reportar las métricas Accuracy, F1-Macro (obligatoria), F1 por clase y la matriz de confusión.
10. Comparar las representaciones clásicas (BoW/TF-IDF) con modelos Transformer.
11. Realizar fine-tuning de un modelo Transformer (BERT/RoBERTa).
12. Definir y reportar hiperparámetros y condiciones de entrenamiento (semillas, batch size, lr, épocas).

Explicabilidad y Análisis de Errores

13. Aplicar explicabilidad local a 20 ejemplos (10 aciertos + 10 errores).
14. Usar para los modelos clásicos top n-grams/coeficientes o SHAP/LIME sobre TF-IDF.
15. Emplear para los Transformer métodos como Integrated Gradients o atribución sobre atención (con cautela).
16. Mostrar fragmentos textuales resaltados (rationales) y una breve justificación de la etiqueta elegida.
17. Analizar los errores típicos (FP/FN) y en subgrupos específicos (longitud, mayúsculas, URLs).
18. Proponer medidas de mitigación para los errores encontrados.

Reproducibilidad y Entrega

19. Garantizar la reproducibilidad fijando semillas.
20. Listar dependencias en un requirements.txt o environment.yml.
21. Crear scripts train.py y eval.py, y guardar el mejor checkpoint (best.ckpt) y configuración (config.json).
22. Comunicar los resultados claramente en un Informe.pdf (8–12 páginas).
23. Incluir un README.md con los pasos exactos para reproducir todos los resultados y tablas.

# Fase 1: Preparación y armonización de datos

## 1. Seleccionar Datasets
Nos centramos en desarrollar la práctica sobre la identificación de géneros musicales a partir de las letras de las canciones (etiquetadas como _lyrics_ en nuestros datasets). Encontramos los datasets buscando por internet y HuggingFace, encontrando uno a través del siguiente enlace: https://data.mendeley.com/datasets/3t9vbwxgr5/2 y el otro en HuggingFace en este enlace: https://www.kaggle.com/datasets/carlosgdcj/genius-song-lyrics-with-language-information/data.

Pero investigando un poco más descubrimos que el primero se encotraba también en HuggingFace, lo que nos simplificó la descarga de los datos, que ahora se consiguen llamando a la función `download_datasets()` de `src/utils/io.py`.

In [19]:
from src.utils.io import DatasetManager

dm = DatasetManager()
dm.download(df1=True, df2=False, df3=False)

Directorio de destino deseado para CSVs: C:\Users\diego\OneDrive - Universidad Rey Juan Carlos\Documentos\GIA_URJC\Curso 2025-26\PLN2\Practicas\practica1\pln2_practica1\data

Descargando saurabhshahane/music-dataset-1950-to-2019...


100%|██████████| 9.73M/9.73M [00:01<00:00, 9.67MB/s]

Extracting files...


Descarga inicial completada en: C:\Users\diego\OneDrive - Universidad Rey Juan Carlos\Documentos\GIA_URJC\Curso 2025-26\PLN2\Practicas\practica1\pln2_practica1\data\datasets\saurabhshahane\music-dataset-1950-to-2019\versions\3
Moviendo 'tcc_ceds_music.csv' a 'C:\Users\diego\OneDrive - Universidad Rey Juan Carlos\Documentos\GIA_URJC\Curso 2025-26\PLN2\Practicas\practica1\pln2_practica1\data\tcc_ceds_music.csv'...
Error al descargar o procesar saurabhshahane/music-dataset-1950-to-2019: [Errno 22] Invalid argument: 'C:\\Users\\diego\\OneDrive - Universidad Rey Juan Carlos\\Documentos\\GIA_URJC\\Curso 2025-26\\PLN2\\Practicas\\practica1\\pln2_practica1\\data\\tcc_ceds_music.csv'

Limpiando directorio de caché de kagglehub: C:\Users\diego\OneDrive - Universidad Rey Juan Carlos\Documentos\GIA_URJC\Curso 2025-26\PLN2\Practicas\practica1\pln2_practica1\data\datasets
Limpieza completada.


## 2. Cargamos el dataset `song_lyrics.csv` nos quedamos con el idioma que nos interesa y lo guardamos

In [20]:
df = dm.load("song_lyrics.csv")

`HAY QUE MODIFICAR ESTE CÓDIGO PARA QUE APAREZCA BONITO Y ELIMINAR EL ERROR DE LA CELDA DE ABAJO`

In [22]:
# Filtrar únicamente las filas donde 'languaje' == 'en'
df_filtrado = df[df["language"] == "en"]

# Guardar el dataset en el directorio /data
# dm.save_dataset(df_filtrado, "en_song_lyrics.csv") # Sale el error de que la función no existe

# Guarda los datos en vez de usar la función de arriba
df_filtrado.to_csv("en_song_lyrics.csv", index=False)

## 3. SELECCIÓN Y ARMONIZACIÓN DE DATOS:

In [28]:
from src.utils.io import DatasetManager

dm = DatasetManager()
df = dm.load("en_song_lyrics.csv")
df.shape

Directorio de destino deseado para CSVs: C:\Users\diego\OneDrive - Universidad Rey Juan Carlos\Documentos\GIA_URJC\Curso 2025-26\PLN2\Practicas\practica1\pln2_practica1\data


(3374198, 11)

In [34]:
df["tag"].unique()

array(['rap', 'rb', 'rock', 'pop', 'misc', 'country'], dtype=object)

In [32]:
from src.utils.preprocesing import DataProcessor

dp = DataProcessor(dm)

# Ejecutar pipeline
dp.load_and_unify("en_song_lyrics.csv") # Puntos 1 y 2
dp.normalize_labels()                   # Punto 3
dp.clean_and_deduplicate()              # Punto 4
dp.handle_imbalance()                   # Punto 5

df = dp.get_df()

Dataset cargado: (3374198, 11)

Estandarizando nombres de etiquetas...
Eliminando 140986 filas (misc/desconocidos).
Asignando IDs numéricos: {'Country': 0, 'Hip Hop': 1, 'Pop': 2, 'R&B': 3, 'Rock': 4}
Clases resultantes: []

Limpiando y deduplicando...
--> Filas eliminadas: 0
--> Dataset limpio: 0

Gestionando desbalance de clases...
Distribución original:
Series([], Name: count, dtype: int64)


ValueError: No objects to concatenate